In [ ]:
import torch

print("Pytorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU",torch.cuda.get_device_name(0))

device="cuda" if torch.cuda.is_available() else "cpu"

print("Using device:" , device)




In [ ]:
!nvidia-smi

In [ ]:
%pip install -q \
    transformers \
    datasets \
    accelerate \
    evaluate \
    jiwer \
    librosa \
    soundfile \
    huggingface_hub

In [ ]:
import transformers
import datasets
import accelerate
import evaluate
import jiwer
import librosa
import soundfile
import huggingface_hub

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("Evaluate:", evaluate.__version__)

print("Speech environment setup complete")

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "ai4bharat/IndicVoices",
    "telugu",
    split="valid",
    streaming=True
)

print(dataset)

In [ ]:
sample = next(iter(dataset))
print(sample)

In [ ]:
audio=sample["audio_filepath"]
audio_samples =  audio.get_all_samples()

waveform=audio_samples.data
sample_rate=audio_samples.sample_rate

print("Waveform shape:", waveform.shape)
print("Sample rate:", sample_rate)
print("Number of samples:", waveform.shape[-1])

In [ ]:
from IPython.display import Audio, display

display(
    Audio(
        waveform.squeeze().cpu().numpy(),
        rate=sample_rate
    )
)

In [ ]:
model_url = "https://indicwhisper.objectstore.e2enetworks.net/telugu_models.zip"
model_zip = "/content/telugu_models.zip"
model_dir = "/content/indicwhisper_telugu"

In [ ]:
!wget -q --show-progress "$model_url" -O "$model_zip"

In [ ]:
import os 
size_gb =os.path.getsize(model_zip) / (1024**3)

print(f"Downloaded archive size: {size_gb:.2f} GB")

In [ ]:
import zipfile

with zipfile.ZipFile(model_zip, 'r') as z:
    files = z.namelist()

print("Number of files:", len(files))

for file in files[:30]:
    print(file)

In [ ]:
import os 
import zipfile

with zipfile.ZipFile(model_zip, 'r') as z:
    z.extractall(model_dir)

checkpoint_dir= os.path.join(
    model_dir,
    "telugu_models",
    "whisper-medium-te_alldata_multigpu"
)

print("Checkpoint directory:",checkpoint_dir)
print("Exists:",os.path.exists(checkpoint_dir))


In [ ]:
important_files = [
    "config.json",
    "generation_config.json",
    "preprocessor_config.json",
    "tokenizer_config.json",
    "vocab.json",
    "merges.txt",
    "pytorch_model.bin",
]

for filename in important_files:
    path = os.path.join(checkpoint_dir, filename)
    print(filename, "->", os.path.exists(path))

In [ ]:
weights_path = os.path.join(
    checkpoint_dir,
    "pytorch_model.bin"
)

weights_size_gb = os.path.getsize(weights_path) / (1024 ** 3)

print(f"Model weights size: {weights_size_gb:.2f} GB")

In [ ]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(checkpoint_dir)

print(processor)

In [ ]:
audio_array=waveform.squeeze().cpu().numpy()

inputs = processor(
    audio_array,
    sampling_rate = sample_rate,
    return_tensors="pt"


)
print("Input features shape:", inputs.input_features.shape)
print("Input features dtype:", inputs.input_features.dtype)

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained(
    checkpoint_dir,
    dtype=torch.float16,
    low_cpu_mem_usage=True
)

model = model.to(device)
model.eval()

In [ ]:
num_params = sum(p.numel() for p in model.parameters())

print(f"Parameters: {num_params / 1e6:.1f} million")
print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)

In [ ]:
gpu_memory_gb = torch.cuda.memory_allocated() / (1024 ** 3)

print(f"GPU memory currently allocated: {gpu_memory_gb:.2f} GB")

In [ ]:
forced_decoder_ids = processor.tokenizer.get_decoder_prompt_ids(
    language="te",
    task="transcribe"
)

print(forced_decoder_ids)

In [ ]:
input_features = inputs.input_features.to(
    device=device,
    dtype=torch.float16
)

print("Features device:", input_features.device)
print("Features dtype:", input_features.dtype)

In [ ]:
with torch.inference_mode():
    predicted_ids = model.generate(
        input_features,
        forced_decoder_ids=forced_decoder_ids
    )

In [ ]:
print("Predicted token IDs:")
print(predicted_ids)

In [ ]:
prediction = processor.batch_decode(
    predicted_ids,
    skip_special_tokens=True
)[0].strip()

reference = sample["normalized"]

print("Reference :", reference)
print("Prediction:", prediction)